# Advanced: Lambda + API Gateway Polly Pipeline

This notebook deploys the `handler.py` Lambda function code and invokes it via an existing API Gateway endpoint.

```
This notebook  →  Lambda (PollyTextToSpeech_Beta/Prod)  →  S3: polly-audio/{env}/{timestamp}.mp3
```

> Running inside SageMaker — credentials come from the instance role automatically.

**Prerequisites — these must already exist in AWS:**
- Lambda functions: `PollyTextToSpeech_Beta` and `PollyTextToSpeech_Prod`
- Each fronted by an API Gateway with `POST /synthesize`
- Lambda execution roles with `polly:SynthesizeSpeech` and `s3:PutObject`
- Your SageMaker role needs `lambda:UpdateFunctionCode` to deploy code updates

## 1. Configuration

In [ ]:
# ── Set these before running ─────────────────────────────────────────────────
AWS_REGION        = "us-east-1"
S3_BUCKET_NAME    = "your-bucket-name"                          # <-- replace
BETA_API_ENDPOINT = "https://your-beta-api-id.execute-api.us-east-1.amazonaws.com/beta/synthesize"  # <-- replace
PROD_API_ENDPOINT = "https://your-prod-api-id.execute-api.us-east-1.amazonaws.com/prod/synthesize"  # <-- replace
# ─────────────────────────────────────────────────────────────────────────────

BETA_FUNCTION_NAME = "PollyTextToSpeech_Beta"
PROD_FUNCTION_NAME = "PollyTextToSpeech_Prod"

## 2. Deploy Lambda Code

Packages `handler.py` and pushes it to both Lambda functions.

In [ ]:
# The Lambda handler — same logic as advanced/lambda/handler.py
HANDLER_CODE = '''
import boto3
import json
import os
from datetime import datetime, timezone

def lambda_handler(event, context):
    environment = os.environ.get("ENVIRONMENT", "beta")
    bucket = os.environ["S3_BUCKET_NAME"]

    body = json.loads(event.get("body") or "{}")
    text = body.get("text", "").strip()

    if not text:
        return {"statusCode": 400, "body": json.dumps({"error": "Missing 'text' in request body"})}

    polly = boto3.client("polly")
    response = polly.synthesize_speech(
        Text=text,
        OutputFormat="mp3",
        VoiceId="Joanna",
        Engine="neural",
    )

    audio = response["AudioStream"].read()
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    key = f"polly-audio/{environment}/{timestamp}.mp3"

    s3 = boto3.client("s3")
    s3.put_object(Bucket=bucket, Key=key, Body=audio, ContentType="audio/mpeg")

    return {
        "statusCode": 200,
        "body": json.dumps({"message": "Audio synthesized", "s3_key": key}),
    }
'''

In [ ]:
import boto3, io, zipfile

def build_zip(code: str) -> bytes:
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("handler.py", code)
    return buf.getvalue()

lambda_client = boto3.client("lambda", region_name=AWS_REGION)
zip_bytes = build_zip(HANDLER_CODE)

for fn in [BETA_FUNCTION_NAME, PROD_FUNCTION_NAME]:
    lambda_client.update_function_code(FunctionName=fn, ZipFile=zip_bytes)
    print(f"Deployed code to {fn}")

## 3. Invoke via API Gateway

Choose which environment to test by setting `ENVIRONMENT`.

In [ ]:
import json, urllib.request

# Switch between "beta" and "prod"
ENVIRONMENT = "beta"   # <-- change to "prod" for production

endpoint = BETA_API_ENDPOINT if ENVIRONMENT == "beta" else PROD_API_ENDPOINT
payload  = json.dumps({"text": "This is from a Lambda via SageMaker."}).encode()

req = urllib.request.Request(
    endpoint,
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST",
)

with urllib.request.urlopen(req) as resp:
    result = json.loads(resp.read())

print(f"Response: {result}")
print(f"Audio saved to: s3://{S3_BUCKET_NAME}/{result.get('s3_key', '')}")

## 4. Verify Output in S3

In [ ]:
s3 = boto3.client("s3", region_name=AWS_REGION)

for prefix in ["polly-audio/beta/", "polly-audio/prod/"]:
    resp = s3.list_objects_v2(Bucket=S3_BUCKET_NAME, Prefix=prefix)
    files = [o["Key"] for o in resp.get("Contents", [])]
    print(f"{prefix}: {len(files)} file(s)")
    for f in files[-3:]:  # show last 3
        print(f"  {f}")